# Classify 100K Customer Questions with JEV

This notebook processes a large CSV in manageable batches. Each JEV request contains multiple independent questions, and several batches can run concurrently. Results are written incrementally to a new CSV to keep memory usage controlled.

In [ ]:
!pip install -q typesafe-sdk

## API key

In Colab, open **Secrets**, add `TYPESAFE_API_KEY`, and enable notebook access.

In [ ]:
from google.colab import userdata
from typesafe_sdk import AsyncTypeSafeClient, Choice
from collections import Counter
import asyncio
import csv
import os

os.environ["TYPESAFE_API_KEY"] = userdata.get("TYPESAFE_API_KEY")

## Configuration

Upload your CSV to Colab and update `INPUT_FILE` if its filename is different. The CSV must contain a column named `question`.

In [ ]:
INPUT_FILE = "/content/customer_questions_100000.csv"
OUTPUT_FILE = "/content/customer_questions_classified.csv"

BATCH_SIZE = 50
MAX_CONCURRENT_BATCHES = 10

In [ ]:
def create_batches(reader, batch_size):
    batch = []

    for row_number, row in enumerate(reader):
        batch.append({
            "id": row_number,
            "question": row["question"]
        })

        if len(batch) == batch_size:
            yield batch
            batch = []

    if batch:
        yield batch


async def classify_batch(client, batch, semaphore):
    questions = {}

    for record in batch:
        question_id = f"row_{record['id']}"
        questions[question_id] = Choice(
            instructions={
                "customer_question": record["question"],
                "task": (
                    "Classify `customer_question` into the most "
                    "appropriate support category."
                )
            },
            criteria={
                "billing": (
                    "Questions about payments, charges, refunds, "
                    "invoices, or subscriptions."
                ),
                "technical_support": (
                    "Questions about technical problems, errors, "
                    "bugs, login issues, or troubleshooting."
                ),
                "sales": (
                    "Questions about pricing, plans, demos, "
                    "purchasing, or enterprise options."
                ),
                "other": (
                    "Questions that do not belong to billing, "
                    "technical support, or sales."
                )
            }
        )

    async with semaphore:
        response = await client.system_one(
            state="Classify each customer-support question independently.",
            questions=questions
        )

    results = []

    for record in batch:
        question_id = f"row_{record['id']}"
        answer = response.choices[question_id]
        results.append({
            "id": record["id"],
            "question": record["question"],
            "category": answer.choice,
            "confidence": answer.confidence
        })

    return results

In [ ]:
async def main():
    semaphore = asyncio.Semaphore(MAX_CONCURRENT_BATCHES)
    category_counts = Counter()
    processed = 0

    with open(INPUT_FILE, "r", encoding="utf-8", newline="") as input_file, \
         open(OUTPUT_FILE, "w", encoding="utf-8", newline="") as output_file:

        reader = csv.DictReader(input_file)
        if not reader.fieldnames or "question" not in reader.fieldnames:
            raise ValueError("The input CSV must contain a 'question' column.")

        writer = csv.DictWriter(
            output_file,
            fieldnames=["id", "question", "category", "confidence"]
        )
        writer.writeheader()

        async with AsyncTypeSafeClient(timeout=120.0) as client:
            pending_batches = []

            async def save_pending():
                nonlocal processed, pending_batches
                completed_batches = await asyncio.gather(*pending_batches)

                for results in completed_batches:
                    for result in results:
                        writer.writerow(result)
                        category_counts[result["category"]] += 1
                        processed += 1

                output_file.flush()
                print(f"Processed {processed:,} records")
                pending_batches = []

            for batch in create_batches(reader, BATCH_SIZE):
                pending_batches.append(
                    classify_batch(client, batch, semaphore)
                )

                if len(pending_batches) == MAX_CONCURRENT_BATCHES:
                    await save_pending()

            if pending_batches:
                await save_pending()

    return category_counts


category_counts = await main()

print("\nClassification complete")
print(f"Results saved to: {OUTPUT_FILE}")
print("\nCategory counts:")

for category, count in category_counts.items():
    print(f"{category}: {count:,}")

## Download the classified CSV

In [ ]:
from google.colab import files
files.download(OUTPUT_FILE)